<a href="https://colab.research.google.com/github/Sambarlasagna/Deep_reinf_learning/blob/main/Pandareachdense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
!pip install stable-baselines3[extra]
!pip install gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 4.8 MB/s eta 0:00:00


In [ ]:
!pip install huggingface_sb3
!pip install huggingface_hub
!pip install panda_gym

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 12.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873172 sha256=63ee6768d6631dfdfe0104b0fa2d1001fccf7c4e2c7063928793cb965e4f8e61
  Stored in directory: /root/.cache/pip/wheels/72/95/1d/b336e5ee612ae9a019bfff4dc0bedd100ee6f0570db205fdf8
Successfully built pybullet


In [ ]:
import os

import gymnasium as gym
import panda_gym

from huggingface_sb3 import load_from_hub, package_to_hub

from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env

from huggingface_hub import notebook_login

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
env_id = "PandaReachDense-v3"

# Create the env
env = gym.make(env_id)

# Get the state space and action space
s_size = env.observation_space.shape
a_size = env.action_space

In [ ]:
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

The State Space is:  None
Sample observation {'achieved_goal': array([-3.6747253,  1.3604611,  6.681171 ], dtype=float32), 'desired_goal': array([ 7.626978 ,  6.2439976, -3.9241753], dtype=float32), 'observation': array([-5.039571  ,  4.114412  , -8.569052  , -0.35710433, -5.1575074 ,
        3.0572116 ], dtype=float32)}


In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

The Action Space is:  Box(-1.0, 1.0, (3,), float32)
Action Space Sample [-0.52036     0.27855465 -0.7047386 ]


In [ ]:
env = make_vec_env(env_id, n_envs=4)

env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

In [ ]:
model = A2C(policy = "MultiInputPolicy",
            env = env,
            verbose=1)

Using cuda device


In [ ]:
model.learn(1_000_000)

Streaming output truncated to the last 5000 lines.
|    std                | 0.395    |
|    value_loss         | 0.000267 |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.77     |
|    ep_rew_mean        | -0.213   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 417      |
|    iterations         | 23800    |
|    time_elapsed       | 1139     |
|    total_timesteps    | 476000   |
| train/                |          |
|    entropy_loss       | -1.4     |
|    explained_variance | 0.956    |
|    learning_rate      | 0.0007   |
|    n_updates          | 23799    |
|    policy_loss        | -0.0114  |
|    std                | 0.396    |
|    value_loss         | 0.000301 |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.7      |
|    ep_rew_mean        

In [ ]:
# Save the model and  VecNormalize statistics when saving the agent
model.save("a2c-PandaReachDense-v3")
env.save("vec_normalize.pkl")

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.vec_env.vec_video_recorder import VecVideoRecorder
import gymnasium as gym
import os
from IPython.display import HTML, display
from base64 import b64encode

# Load the saved statistics
# Create a new environment for evaluation and video recording
# Wrap the environment with VecVideoRecorder after VecNormalize

eval_env = DummyVecEnv([lambda: gym.make("PandaReachDense-v3", render_mode="rgb_array")])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

# Wrap the normalized vectorized environment with VecVideoRecorder
# We record only the first episode (x == 0) for a length of 500 steps.
eval_env = VecVideoRecorder(eval_env, video_folder="./videos",
                            record_video_trigger=lambda x: x == 0, video_length=500)

# do not update them at test time
eval_env.training = False
# reward normalization is not needed at test time
eval_env.norm_reward = False

# Load the agent
model = A2C.load("a2c-PandaReachDense-v3")

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

# Close the video recorder properly
eval_env.close()

# Display the recorded video
video_path = "./videos/rl-video-step-0-to-step-500.mp4" # Corrected filename based on VecVideoRecorder output

if os.path.exists(video_path):
    with open(video_path, "rb") as f:
        mp4 = f.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'<video width="600" controls><source src="{data_url}" type="video/mp4"></video>'))
else:
    print(f"Video file not found at {video_path}")

Mean reward = -0.13 +/- 0.06
Moviepy - Building video /content/videos/rl-video-step-0-to-step-500.mp4.
Moviepy - Writing video /content/videos/rl-video-step-0-to-step-500.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/rl-video-step-0-to-step-500.mp4


In [ ]:
notebook_login()
!git config --global credential.helper store

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
from huggingface_sb3 import package_to_hub

package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env,
    repo_id=f"Sambarlasagna/a2c-{env_id}",
    commit_message="Panda reach dense",
)

ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Saving video to /tmp/tmpp2lz4jo7/-step-0-to-step-1000.mp4


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Moviepy - Building video /tmp/tmpp2lz4jo7/-step-0-to-step-1000.mp4.
Moviepy - Writing video /tmp/tmpp2lz4jo7/-step-0-to-step-1000.mp4



...

  ...aReachDense-v3/policy.pth: 100%|##########| 46.7kB / 46.7kB            

  ...2c-PandaReachDense-v3.zip: 100%|##########|  114kB /  114kB            

  ...5tqbkl0/vec_normalize.pkl: 100%|##########| 2.64kB / 2.64kB            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/Sambarlasagna/a2c-PandaReachDense-v3/tree/main/


CommitInfo(commit_url='https://huggingface.co/Sambarlasagna/a2c-PandaReachDense-v3/commit/cc69173b7ccd490786d2743c9d98411a398f6a63', commit_message='Panda reach dense', commit_description='', oid='cc69173b7ccd490786d2743c9d98411a398f6a63', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sambarlasagna/a2c-PandaReachDense-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='Sambarlasagna/a2c-PandaReachDense-v3'), pr_revision=None, pr_num=None)

In [ ]:
!mlagents-learn config/ppo/Pyramids.yaml \
  --env=training-envs-executables/linux/Pyramids/Pyramids.x86_64 \
  --run-id=eval_video \
  --inference \
  --no-graphics \
  --record


/bin/bash: line 1: mlagents-learn: command not found
